# basic infomap function 😭

In [1]:
import igraph as ig
import numpy as np

In [2]:
# shannon entropy
# H = - sum(p log2 p)

def entropy(probs):

    probs = np.array(probs)
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

## map equation for unw + und

In [7]:
def map_equation(g, partition):

    # nr of edges
    m = g.ecount()

    # node visit probabilities p_i
    # p_i = degree(i) / (2m)
    degrees = np.array(g.degree())
    p = degrees / (2 * m)

    # nr of communities
    num_comms = max(partition) + 1

    # community visit probabilities p_c
    p_module = np.zeros(num_comms)
    for i, c in enumerate(partition):
        p_module[c] += p[i]

    # exit probabilities q_c
    q = np.zeros(num_comms)

    edges = g.get_edgelist()

    for (i, j) in edges:
        ci = partition[i]
        cj = partition[j]

        # edge crosses communities -> counts as exit 
        # if the edge connects two communities, exits from both
        if ci != cj:
            q[ci] += 1
            q[cj] += 1

    # normalize
    q = q / (2 * m)

    # total exit probability
    q_total = np.sum(q)

    # entropy between communities H(Q)
    if q_total > 0:
        H_Q = entropy(q / q_total)
    else:
        H_Q = 0

    # map equation
    L = q_total * H_Q

    # entropy within each community
    for c in range(num_comms):

        # nodes in community c
        nodes_in_c = [i for i in range(len(partition)) if partition[i] == c]

        # probabilities inside community: node probs + exit prob
        probs = list(p[nodes_in_c])
        probs.append(q[c])

        total = sum(probs)

        if total > 0:
            probs = [x / total for x in probs]
            H_c = entropy(probs)

            L += (p_module[c] + q[c]) * H_c

    return L

## map equation for w + und

In [8]:
def map_equation_weighted(g, partition):
    # insetad of degree -> weighted strength
    if "weight" in g.edge_attributes():
        strength = np.array(g.strength(weights="weight"), dtype=float)
        weights = {(e.source, e.target): e["weight"] for e in g.es}
    else:
        strength = np.array(g.degree(), dtype=float)
        weights = None

    total_strength = strength.sum()  # isntead of 2m
    p = strength / total_strength

    num_comms = max(partition) + 1
    p_module = np.zeros(num_comms)
    for i, c in enumerate(partition):
        p_module[c] += p[i]

    q = np.zeros(num_comms)
    for e in g.es:
        i, j = e.source, e.target
        ci, cj = partition[i], partition[j]
        if ci != cj:
            w = e["weight"] if weights else 1.0
            q[ci] += w   # ← weighted αντί για +1
            q[cj] += w

    q = q / total_strength  

    # for H is the same with unweighted
    q_total = np.sum(q)
    H_Q = entropy(q / q_total) if q_total > 0 else 0
    L = q_total * H_Q

    for c in range(num_comms):
        nodes_in_c = [i for i in range(len(partition)) if partition[i] == c]
        probs = list(p[nodes_in_c]) + [q[c]]
        total = sum(probs)
        if total > 0:
            probs = [x / total for x in probs]
            L += (p_module[c] + q[c]) * entropy(probs)

    return L

## test on lauras weighted + undirected

In [5]:
# weighted undirected
g3 = ig.Graph.Read_GraphML("C:/Users/savin/ITI/weighted_undirected.graphml")
partition3 = [int(v["community"]) for v in g3.vs]

# length
L_custom_w = map_equation_weighted(g3, partition3)

# compare igraph
communities3 = g3.community_infomap(edge_weights="weight")
L_igraph_w = communities3.codelength

print("Custom weighted L: ", L_custom_w)
print("igraph weighted L: ", L_igraph_w)

Custom weighted L:  3.4884048364539413
igraph weighted L:  3.336712472760091


## test on lauras unweighted + undirected

In [9]:
# unweighted + undirectedd
g = ig.Graph.Read_GraphML("C:/Users/savin/ITI/unweighted_undirected.graphml")

# partition from ground truth (attribute "community")
partition = [int(v["community"]) for v in g.vs]

# compute description length
L_custom = map_equation(g, partition)

# compare with igraph Infomap
communities = g.community_infomap()
L_igraph = communities.codelength

print("Custom L:", L_custom)
print("igraph L:", L_igraph)

Custom L: 3.4015829735877694
igraph L: 3.40158297358777


In [10]:
infomap_partition = communities3.membership

In [11]:
L_custom_on_infomap = map_equation_weighted(g3, infomap_partition)

In [12]:
print("Custom L (Infomap partition):", L_custom_on_infomap)
print("Infomap L:", communities3.codelength)

Custom L (Infomap partition): 3.336712472760091
Infomap L: 3.336712472760091
